# P1-23 · Score all saved Paper 1 runs against manual breast/pectoral annotations of the 84 test images (CPU, ~10 min)
**Runtime: CPU.** No training. Read-only for QSEG_FINAL and all run folders. Writes only `MyDrive/P1_v2/annot_results/`.
Annotations: LabelMe JSON files (polygon labels `breast` and `pectoral`, drawn on the full-resolution original images, saved with `--nodata`) in
`MyDrive/P1_v2/annot_test/reader1/` (all 84) and optionally `reader2/` (second reader, subset).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, json, numpy as np, pandas as pd, cv2
ROOT = '/content/drive/MyDrive/QSEG_FINAL'; OUT = '/content/drive/MyDrive/P1_v2'
RES = f'{OUT}/annot_results'; os.makedirs(RES, exist_ok=True)
import sys; sys.path.insert(0, ROOT + '/code')
from qseg.data_prep import generate_breast_mask, pad_to_square, chest_wall_side, canonical_id
man = pd.read_csv(ROOT + '/results/data/manifest.csv'); test = man[man.split == 'test'].reset_index(drop=True)
print('test images:', len(test), '| patients:', test.patient_id.nunique())
raw = {}
for f in test.original_filename:
    hits = glob.glob(f'{ROOT}/**/{glob.escape(f)}', recursive=True)
    hits = [h for h in hits if '/results/' not in h]
    raw[f] = hits[0] if hits else None
print('original files found:', sum(v is not None for v in raw.values()), 'of', len(raw)); assert all(raw.values())

In [ ]:
def load_reader(r):
    out = {}
    for p in sorted(glob.glob(f'{OUT}/annot_test/{r}/*.json')):
        j = json.load(open(p)); fid = canonical_id(re.split(r'[\\/]', j['imagePath'])[-1])  # imagePath may hold Windows separators
        assert fid == canonical_id(os.path.basename(p)), (p, j['imagePath'])
        out[fid] = j
    return out
READERS = {r: load_reader(r) for r in ('reader1', 'reader2') if os.path.isdir(f'{OUT}/annot_test/{r}')}
for r, d in READERS.items():
    print(r, len(d), 'files | not in test set:', sorted(set(d) - set(test.file_id))[:5])
missing = sorted(set(test.file_id) - set(READERS['reader1']))
print('reader1 missing:', len(missing), missing[:10])

In [ ]:
def raster(j, shape):
    H, W = shape; m = {'breast': np.zeros((H, W), np.uint8), 'pectoral': np.zeros((H, W), np.uint8)}
    bad = []
    for s in j['shapes']:
        lab = s['label'].strip().lower()
        if lab not in m or s.get('shape_type', 'polygon') != 'polygon': bad.append(lab); continue
        cv2.fillPoly(m[lab], [np.round(np.array(s['points'])).astype(np.int32)], 1)
    return m, bad
def to256(mask, side):
    return (cv2.resize(pad_to_square(mask.astype(np.uint8) * 255, side), (256, 256), interpolation=cv2.INTER_NEAREST) > 127)
def dice(a, b):
    s = a.sum() + b.sum(); return 1.0 if s == 0 else 2 * (a & b).sum() / s
A, QC = {}, []
for _, row in test.iterrows():
    gray = cv2.imread(raw[row.original_filename], cv2.IMREAD_GRAYSCALE)
    proxy, _, _ = generate_breast_mask(gray, 20)
    side = 'left' if chest_wall_side(proxy) == 'right' else 'right'
    saved = cv2.imread(f'{ROOT}/results/data/masks/{row.file_id}.png', 0) > 127
    regen = to256(proxy > 0, side)
    for r, d in READERS.items():
        if row.file_id not in d: continue
        j = d[row.file_id]
        assert (j['imageHeight'], j['imageWidth']) == gray.shape, (row.file_id, j['imageHeight'], j['imageWidth'], gray.shape)
        m, bad = raster(j, gray.shape)
        full = (m['breast'] | m['pectoral']) > 0; br = (m['breast'] > 0) & ~(m['pectoral'] > 0)
        A[(r, row.file_id)] = dict(full=to256(full, side), breast=to256(br, side))
        QC.append(dict(reader=r, file_id=row.file_id, view=row['view'], bad_labels=';'.join(bad), has_pectoral=bool(m['pectoral'].any()),
                       frac_full=float(full.mean()), proxy_regen_vs_saved=dice(regen, saved),
                       rule_vs_annot_full=dice(saved, A[(r, row.file_id)]['full']), rule_vs_annot_breast=dice(saved, A[(r, row.file_id)]['breast'])))
QC = pd.DataFrame(QC); QC.to_csv(f'{RES}/annot_qc.csv', index=False)
print(QC.groupby('reader')[['proxy_regen_vs_saved', 'rule_vs_annot_full', 'rule_vs_annot_breast']].describe().T.round(4).to_string())
print('unexpected labels:', QC.bad_labels[QC.bad_labels.fillna('') != ''].unique()[:10]); print('MLO without pectoral:', QC[(QC.view == 'MLO') & ~QC.has_pectoral].file_id.tolist())

In [ ]:
rows = []
for d in sorted(glob.glob(f'{OUT}/runs/*_seed*')):
    f = f'{d}/test_prob_uint8.npz'
    if not os.path.exists(f'{d}/run.json') or not os.path.exists(f): continue
    z = np.load(f, allow_pickle=True); P = z['prob'] >= 128; fids = [str(x) for x in z['file_id']]
    name = os.path.basename(d); exp, mdl, s = name.split('_')
    for k, fid in enumerate(fids):
        for r in READERS:
            if (r, fid) not in A: continue
            rows.append(dict(run=name, exp=exp, model=mdl, seed=int(s[4:]), reader=r, file_id=fid,
                             dice_full=dice(P[k], A[(r, fid)]['full']), dice_breast=dice(P[k], A[(r, fid)]['breast'])))
S = pd.DataFrame(rows).merge(test[['file_id', 'patient_id']], on='file_id')
S.to_csv(f'{RES}/annot_scores.csv.gz', index=False)
print(S.groupby(['reader', 'exp', 'model'])[['dice_full', 'dice_breast']].mean().round(4).to_string())
if 'reader2' in READERS:
    ir = [dict(file_id=f, dice_full=dice(A[('reader1', f)]['full'], A[('reader2', f)]['full']), dice_breast=dice(A[('reader1', f)]['breast'], A[('reader2', f)]['breast']))
          for (r, f) in A if r == 'reader2' and ('reader1', f) in A]
    ir = pd.DataFrame(ir); ir.to_csv(f'{RES}/inter_reader.csv', index=False); print('inter-reader:', ir[['dice_full', 'dice_breast']].describe().round(4).to_string())
print('DONE')

In [ ]:
# QC sheet: annotation outlines (green breast, orange pectoral) on the 256 x 256 inputs, reader1 (stays in Drive)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
ids = [f for (r, f) in A if r == 'reader1']
for p in range(0, len(ids), 28):
    fig, ax = plt.subplots(4, 7, figsize=(17, 10))
    for a, f in zip(ax.ravel(), ids[p:p + 28]):
        im = cv2.cvtColor(cv2.imread(f'{ROOT}/results/data/images/{f}.png', 0), cv2.COLOR_GRAY2RGB)
        for key, col in (('full', (60, 220, 60)), ('breast', (255, 160, 0))):
            cs, _ = cv2.findContours(A[('reader1', f)][key].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(im, cs, -1, col, 1)
        a.imshow(im); a.set_title(f, fontsize=6)
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.savefig(f'{RES}/qc_sheet_{p // 28 + 1}.png', dpi=110); plt.close()
print('QC sheets saved')